Week 14 · Day 6 — Inference & Packaging
Why this matters

Training is only half the story — to use your fine-tuned model in apps, you need a clean inference pipeline: load model + tokenizer, preprocess input, run prediction, return output. Packaging ensures reproducibility and deployability.

Theory Essentials

Save & Load: trainer.save_model() and from_pretrained() make models portable.

Inference = tokenize → model → decode/argmax.

Determinism: set seeds for reproducible results.

Deployment: bundle model + tokenizer + preprocessing code.

Efficiency: batch inputs, use CPU/GPU accordingly.

In [1]:
# =========================
# Setup
# =========================
import os, random, numpy as np, torch
from pathlib import Path
import matplotlib.pyplot as plt
np.random.seed(42); random.seed(42); torch.manual_seed(42)
plt.rcParams["figure.figsize"] = (6,4); plt.rcParams["axes.grid"] = True

CLASS_DIR = Path("./finetuned-distilbert-imdb")
GEN_DIR   = Path("./finetuned-gpt2-imdb")

def ensure_dir(d: Path):
    d.mkdir(parents=True, exist_ok=True)

# Optional: keep CPU threads reasonable
try:
    torch.set_num_threads(max(1, os.cpu_count() // 2))
except Exception:
    pass

# =========================
# A) CLASSIFICATION (DistilBERT)
# =========================
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

MAX_LEN_CLS   = 64     # ↓ shorter sequences = faster
N_TRAIN_CLS   = 1000   # ↓ subset size
N_EVAL_CLS    = 400

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}

def maybe_train_classifier():
    if CLASS_DIR.exists() and any(CLASS_DIR.iterdir()):
        print(f"[Classifier] Using existing checkpoint at: {CLASS_DIR}")
        return

    print("[Classifier] Training a small model (CPU-fast)...")
    dataset = load_dataset("imdb")
    tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
    collator = DataCollatorWithPadding(tokenizer=tok)  # dynamic padding

    def tokenize(batch):
        # no padding here; collator pads to longest in batch
        return tok(batch["text"], truncation=True, max_length=MAX_LEN_CLS)

    ds = dataset.map(tokenize, batched=True).remove_columns(["text"])
    ds = ds.rename_column("label", "labels")
    ds.set_format("torch")

    train = ds["train"].shuffle(seed=42).select(range(N_TRAIN_CLS))
    evald = ds["test"].shuffle(seed=42).select(range(N_EVAL_CLS))

    model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
    args = TrainingArguments(
        output_dir="./_tmp_cls",
        per_device_train_batch_size=16,   # can be larger with dynamic padding
        per_device_eval_batch_size=32,
        num_train_epochs=1,               # Day-6 default
        no_cuda=True,                     # ensure CPU
        logging_steps=100
    )
    trainer = Trainer(model=model, args=args, train_dataset=train, eval_dataset=evald,
                      data_collator=collator, compute_metrics=compute_metrics)
    trainer.train()
    print("[Classifier Eval]:", trainer.evaluate())

    ensure_dir(CLASS_DIR)
    trainer.save_model(CLASS_DIR)
    tok.save_pretrained(CLASS_DIR)

def load_classifier():
    tok = AutoTokenizer.from_pretrained(CLASS_DIR)
    model = AutoModelForSequenceClassification.from_pretrained(CLASS_DIR)
    model.eval()
    return tok, model

def predict(texts):
    if isinstance(texts, str): texts = [texts]
    tok, model = load_classifier()
    with torch.no_grad():
        enc = tok(texts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LEN_CLS)
        out = model(**enc)
        probs = torch.softmax(out.logits, dim=-1).numpy()
    return [
        {"text": t, "label": int(p.argmax()), "prob_negative": float(p[0]), "prob_positive": float(p[1])}
        for t, p in zip(texts, probs)
    ]

# =========================
# B) GENERATION (distilGPT-2)
# =========================
from transformers import AutoModelForCausalLM, DataCollatorForLanguageModeling

MAX_LEN_GEN   = 64
N_TRAIN_GEN   = 1000
N_EVAL_GEN    = 200
GPT_MODEL     = "distilgpt2"    # ↓ much faster than "gpt2"

def maybe_train_generator():
    if GEN_DIR.exists() and any(GEN_DIR.iterdir()):
        print(f"[Generator] Using existing checkpoint at: {GEN_DIR}")
        return

    print("[Generator] Training a small GPT (CPU-fast)...")
    dataset = load_dataset("imdb")
    tok = AutoTokenizer.from_pretrained(GPT_MODEL)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    def tokenize(batch):
        # no padding here; the LM collator pads dynamically
        return tok(batch["text"], truncation=True, max_length=MAX_LEN_GEN)

    ds = dataset.map(tokenize, batched=True).remove_columns(["text"])
    ds.set_format("torch")

    train = ds["train"].shuffle(seed=42).select(range(N_TRAIN_GEN))
    evald = ds["test"].shuffle(seed=42).select(range(N_EVAL_GEN))

    model = AutoModelForCausalLM.from_pretrained(GPT_MODEL)
    model.resize_token_embeddings(len(tok))
    model.config.pad_token_id = tok.pad_token_id
    model.config.use_cache = False  # faster/less memory during train

    collator = DataCollatorForLanguageModeling(tokenizer=tok, mlm=False)  # creates labels=input_ids

    args = TrainingArguments(
        output_dir="./_tmp_gen",
        per_device_train_batch_size=8,    # can go higher on CPU with small seqs
        per_device_eval_batch_size=16,
        num_train_epochs=1,
        no_cuda=True,
        logging_steps=100
    )
    trainer = Trainer(model=model, args=args, train_dataset=train, eval_dataset=evald,
                      data_collator=collator)
    trainer.train()

    ensure_dir(GEN_DIR)
    trainer.save_model(GEN_DIR)
    tok.save_pretrained(GEN_DIR)

def load_generator():
    tok = AutoTokenizer.from_pretrained(GEN_DIR)
    model = AutoModelForCausalLM.from_pretrained(GEN_DIR)
    model.eval()
    return tok, model

def generate_text(prompt, max_new_tokens=40, temperature=0.8, top_p=0.95):
    tok, model = load_generator()
    enc = tok(prompt, return_tensors="pt")
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=True,
                         temperature=temperature, top_p=top_p, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0], skip_special_tokens=True)

# =========================
# DEMO
# =========================
maybe_train_classifier()
maybe_train_generator()

print("\n[Demo] Classification:")
for r in predict(["This movie was amazing!", "Worst film I’ve seen."]):
    print(r)

print("\n[Demo] GPT Generation:")
print(generate_text("The movie was"))


[Classifier] Using existing checkpoint at: finetuned-distilbert-imdb
[Generator] Training a small GPT (CPU-fast)...


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

c:\AI-Mastery\venv\Lib\site-packages\transformers\training_args.py:1619: FutureWarning: using `no_cuda` is deprecated and will be removed in version 5.0 of 🤗 Transformers. Use `use_cpu` instead
  warnings.warn(
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
100,4.084600



[Demo] Classification:
{'text': 'This movie was amazing!', 'label': 1, 'prob_negative': 0.08421219140291214, 'prob_positive': 0.9157878160476685}
{'text': 'Worst film I’ve seen.', 'label': 0, 'prob_negative': 0.8539913296699524, 'prob_positive': 0.1460086703300476}

[Demo] GPT Generation:
The movie was a huge success. I loved it. It was one of my favorite films. I could only imagine how I would have liked this movie to be on the same shelf. The plot was very simple and


1) Core (10–15 min)
Task: Run the full code. Call predict() on 3 sample reviews.

In [2]:
for r in predict(["This movie was the worst!", "Horrible satrt but at the end it got much better. I recommend it."]):
    print(r)

{'text': 'This movie was the worst!', 'label': 0, 'prob_negative': 0.8883019089698792, 'prob_positive': 0.11169803887605667}
{'text': 'Horrible satrt but at the end it got much better. I recommend it.', 'label': 0, 'prob_negative': 0.7881240248680115, 'prob_positive': 0.21187591552734375}


2) Practice (10–15 min)
Task: Batch 10 reviews at once. Compare runtime vs running them one by one.

In [4]:
import time

movie_reviews = [
    "This movie was visually stunning but the story felt thin. The ending left me unsatisfied, though the performances were solid.",
    "Absolutely loved it! The pacing, the dialogue, and the soundtrack were all perfect. I’d definitely watch it again.",
    "The film tried to be deep but came across as pretentious. Too many long shots of nothing happening.",
    "A surprisingly fun ride. I went in with low expectations, but the humor and action sequences kept me hooked.",
    "The acting was wooden and the CGI looked fake. Honestly, it felt like a waste of two hours.",
    "Brilliant character development and a plot twist I didn’t see coming. Easily one of my favorites this year.",
    "Mediocre at best. The trailer was more exciting than the full movie.",
    "An emotional rollercoaster. The chemistry between the leads carried the whole story. Bring tissues.",
    "The script was clunky, but the cinematography and set design were breathtaking.",
    "Pure popcorn entertainment. It won’t win any awards, but it kept me smiling the whole time."
]

t0 = time.time()

for r in predict(movie_reviews):
    print(r)
print("Time taken: ", time.time() - t0)

t0 = time.time()
_ = predict(movie_reviews) 
print("Batched time:", time.time() - t0)


{'text': 'This movie was visually stunning but the story felt thin. The ending left me unsatisfied, though the performances were solid.', 'label': 1, 'prob_negative': 0.12327751517295837, 'prob_positive': 0.876722514629364}
{'text': 'Absolutely loved it! The pacing, the dialogue, and the soundtrack were all perfect. I’d definitely watch it again.', 'label': 1, 'prob_negative': 0.08989029377698898, 'prob_positive': 0.9101097583770752}
{'text': 'The film tried to be deep but came across as pretentious. Too many long shots of nothing happening.', 'label': 0, 'prob_negative': 0.8783984184265137, 'prob_positive': 0.12160158902406693}
{'text': 'A surprisingly fun ride. I went in with low expectations, but the humor and action sequences kept me hooked.', 'label': 1, 'prob_negative': 0.15068699419498444, 'prob_positive': 0.8493130207061768}
{'text': 'The acting was wooden and the CGI looked fake. Honestly, it felt like a waste of two hours.', 'label': 0, 'prob_negative': 0.8931397795677185, 'p

3) Stretch (optional, 10–15 min)
Task: Change generation args: temperature=1.0, top_p=0.9. Compare outputs.

In [6]:
print("\n[Demo] GPT Generation: with normal arguments:")
print(generate_text("The movie was"))


print("\n[Demo] GPT Generation: with changed arguments:")
print(generate_text("The movie was", temperature=1.0, top_p=0.9))


[Demo] GPT Generation: with normal arguments:
The movie was very easy to make. It had a good lead, and a great acting (just as it did in this movie). The acting was very nice, and the actors were very good, but the actors

[Demo] GPT Generation: with changed arguments:
The movie was pretty bad and I can only guess what people were thinking. I could not believe the film was so bad at that time.The film was pretty bad and I can only guess what people were thinking.


Mini-Challenge (≤40 min)

Task: Write predict.py and generate.py scripts that load your saved checkpoints and run from CLI. Example:

python predict.py "I loved the soundtrack."
python generate.py "The film reminded me of"


Acceptance Criteria:

Scripts run without retraining.

predict.py → returns {label, prob}.

generate.py → returns a continuation.

Notes / Key Takeaways

Save both model + tokenizer → ensures consistent vocab.

Inference pipeline = preprocess → model → postprocess.

Always separate training and inference.

Batch inputs to reduce latency.

GPT-2 uses causal LM; BERT uses classification head.

Reflection

Why would inference fail if you only saved the model but not the tokenizer?

Why is batching important when deploying to production?

Why would inference fail if you only saved the model but not the tokenizer?
Because the model expects inputs in the exact same tokenized format it was trained with. Without the tokenizer (which defines vocabulary, special tokens, and ID mappings), raw text can’t be correctly converted into token IDs. The result would be mismatched IDs, errors, or meaningless outputs.

Why is batching important when deploying to production?
Batching lets the system process many inputs in one forward pass. This reduces total compute time, improves throughput, and lowers cost. Without batching, each input runs separately, wasting GPU/CPU cycles and slowing down response times under heavy load.